## **Import libraries, module and function**

In [1]:
from torchvision import datasets, transforms

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## **Loading and normalizing the data**


Before training a neural network, the input data needs to be normalized so that the pixel values are scaled into a consistent range. This helps improve stability by keeping input values centered around zero and ensuring that gradients behave more predictably during optimization

In [7]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))

])

In [8]:

train_data = datasets.MNIST(
    root = './data',
    train = True,
    download = True,
    transform = transform
    )

MNIST images are normalized as part of a preprocessing pipeline using PyTorch transforms:

*  `Transforms.ToTensor()` Converts images from pixel values (0-255) into floating-point tesors scaled to 0-1

*   `Transforms.Normalize((0.5,),(0.5,)) ` then rescales these values to approximately -1 to 1, which helps stabilize training by keeping input values centered around zero and improving gradient behavior during optimization.


PyTorch also provides key data-loading parameters to control how training data is processed:




*   `batch_size=64` means ***the model processes 64 images at a time*** instead of the full dataset. This improves memory efficiency and makes training more stable by allowing gradient updates on mini-groups of data rather than individual samples or the entire dataset.
*  `shuffle=True` randomizes the order of images each epoch, so the model does not memorize the sequence.


* `Download = True` means pytorch fetches MNIST automatically on the first run , so you don't need to download anything manually



## **Defining the model**



After the data is ready the next step is to build the neural network that will learn from it. The goal of the model is to take an input image of a handwritten digit and predict which digit (0–9) it represents

In [9]:
from torch.nn import functional
from torch import nn

In [10]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

In [11]:
class SimpleNetwork(nn.Module):
   def __init__(self):
       super(SimpleNetwork, self).__init__()
       self.fc1 = nn.Linear(784, 128)  # 28x28 = 784 input pixels
       self.fc2 = nn.Linear(128, 64)   # hidden layer
       self.fc3 = nn.Linear(64, 10)    # 10 outputs (digits 0-9)


   def forward(self, x):
       x = x.view(-1, 784)             # flatten the image
       x = F.relu(self.fc1(x))
       x = F.relu(self.fc2(x))
       x = self.fc3(x)
       return x


model = SimpleNetwork()
print(model)

SimpleNetwork(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)


## **Training the model**

## **Choosing a loss function and optimizer**

the model is defined, the next step is to ***train it so it can learn to recognize handwritten digits***. During training, the model processes MNIST images,makes predictions, compares them to correct labels, and gradually improves its performance. To do this, we first need two key components: a loss function and an optimiz

## Loss Function

The loss function measures how far the model’s predictions are from the correct answers. In classification problems like MNIST (which has 10 classes, one for each digit),


---


`CrossEntropyLoss` is used because it is designed for multi-classclassification, and it not only penalizes incorrect predictions but also takes into account how confident the model is when it makes a mistake.




---

The optimizer is responsible for updating the model’s weights based on the loss. It determines how the model learns from its errors.

The difference is how they do it. *SGD updates model weights using a fixed learning rate applied to the computed gradients.* **ADAM extends this idea by adapting the learning rate for each parameter using estimates of past gradients, which often leads to faster and more stable convergence with less manual tuning**. For this project, ADAM is the practical choice, with lr=0.001 as a safe default learning rate. SGD is worth exploring later when you want more control over the training process.

## **Implementing a training loop**

**The training loop is the core of the learning process**. Each full pass through the training data is called an epoch. Training typically runs for multiple epochs so that the model can gradually improve its performance over time.


---


Each epoch is made up of smaller units called batches. Instead of processing the entire dataset at once, the model processes one batch at a time, which makes training more efficient and memory-friendly.

During each epoch, the model processes data in batches and repeats the same steps:


* `Forward pass`: The model makes predictions (logits).
* `Loss computation`: The model compares predictions with true labels.
* `Backward pass`: The model computes gradients of the loss.
* `Weight update:` The optimizer adjusts model parameters.

## **Defining the model**



After the data is ready the next step is to build the neural network that will learn from it. The goal of the model is to take an input image of a handwritten digit and predict which digit (0–9) it represents

In [12]:
from torch.nn import functional
from torch import nn

In [13]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

In [14]:
class SimpleNetwork(nn.Module):
   def __init__(self):
       super(SimpleNetwork, self).__init__()
       self.fc1 = nn.Linear(784, 128)  # 28x28 = 784 input pixels
       self.fc2 = nn.Linear(128, 64)   # hidden layer
       self.fc3 = nn.Linear(64, 10)    # 10 outputs (digits 0-9)


   def forward(self, x):
       x = x.view(-1, 784)             # flatten the image
       x = F.relu(self.fc1(x))
       x = F.relu(self.fc2(x))
       x = self.fc3(x)
       return x


model = SimpleNetwork()
print(model)

SimpleNetwork(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)


## **Training the model**

## **Choosing a loss function and optimizer**

the model is defined, the next step is to ***train it so it can learn to recognize handwritten digits***. During training, the model processes MNIST images,makes predictions, compares them to correct labels, and gradually improves its performance. To do this, we first need two key components: a loss function and an optimiz

## Loss Function

The loss function measures how far the model’s predictions are from the correct answers. In classification problems like MNIST (which has 10 classes, one for each digit),


---


`CrossEntropyLoss` is used because it is designed for multi-classclassification, and it not only penalizes incorrect predictions but also takes into account how confident the model is when it makes a mistake.




---

The optimizer is responsible for updating the model’s weights based on the loss. It determines how the model learns from its errors.

The difference is how they do it. *SGD updates model weights using a fixed learning rate applied to the computed gradients.* **ADAM extends this idea by adapting the learning rate for each parameter using estimates of past gradients, which often leads to faster and more stable convergence with less manual tuning**. For this project, ADAM is the practical choice, with lr=0.001 as a safe default learning rate. SGD is worth exploring later when you want more control over the training process.

## **Implementing a training loop**

**The training loop is the core of the learning process**. Each full pass through the training data is called an epoch. Training typically runs for multiple epochs so that the model can gradually improve its performance over time.


---


Each epoch is made up of smaller units called batches. Instead of processing the entire dataset at once, the model processes one batch at a time, which makes training more efficient and memory-friendly.

During each epoch, the model processes data in batches and repeats the same steps:


* `Forward pass`: The model makes predictions (logits).
* `Loss computation`: The model compares predictions with true labels.
* `Backward pass`: The model computes gradients of the loss.
* `Weight update:` The optimizer adjusts model parameters.

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device", device)

model = SimpleNetwork().to(device)

using device cuda


In [16]:
# define loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


#training loop
epochs =5

for epoch in range(epochs):
  model.train()
  running_loss =0

  for images,labels in train_loader:
    #move to GPU/CPU
    images,labels = images.to(device), labels.to(device)
    # forward pass
    prediction = model(images)
    loss = loss_fn(prediction, labels)

    #Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  avg_loss = running_loss / len(train_loader)
  print(f"Epoch {epoch+1}/5 — Loss: {avg_loss:.4f}")

Epoch 1/5 — Loss: 0.4022
Epoch 2/5 — Loss: 0.1946
Epoch 3/5 — Loss: 0.1397
Epoch 4/5 — Loss: 0.1146
Epoch 5/5 — Loss: 0.0953


In [17]:
model.eval()

correct = 0
total = 0

with torch.no_grad(): # no gradient needed
  for images,labels in train_loader:
    images,labels = images.to(device), labels.to(device)

    outputs = model(images)

    #get predicted class
    _, predicted = torch.max(outputs,1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100* correct / total
print(f'test accuracy: {accuracy:.2f}%')


test accuracy: 97.60%
